In [2]:
import numpy as np
import matplotlib.pyplot as plt
import os
from os import listdir
import pandas as pd
from sklearn.linear_model import Lasso, LinearRegression
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [3]:
def t_to_idx(t):
    return int((t-(-2))/0.05)
def z_score(X):
    mean = np.mean(X, axis=0)[np.newaxis,:]
    sd = np.std(X, axis=0)[np.newaxis,:]
    sd[sd==0] = 1
    
    return (X-mean)/sd

In [7]:
subject = 'Tir'

trial_type = 'correct'
path = f'/Users/huidili/Desktop/wmupdate/analysis_data/{trial_type}_trials/'
area_path = path+'area_df/'

for filename in listdir(area_path):
    if ('area' in filename):
        if 'ISA' in filename:
            sub_name, session_num, _, _ = filename.split('_')
            session_name = sub_name+'_'+session_num 
        elif 'Tir' in filename:
            session_name = filename.split('_')[0]

        if (subject == 'Tir' and 'Tir' in session_name) or (subject == 'ISA' and 'ISA' in session_name):            
            condition_path = path+'condition_df/'
            label_path = path+'labels/'

            spk_path = path+'spike_rate/150bin_50step/'
            spk_data = np.load(spk_path+session_name+'_spike_rate.npz')
            spike_rate = spk_data['spike_rate']

            area_df = pd.read_pickle(area_path+session_name+'_area_df.pkl')  
            condition_df = pd.read_pickle(condition_path+session_name+'_condition_df.pkl')
            label_data = np.load(label_path+session_name+'_labels.npy')

            # select PFC neurons and intermediate T1 trials for selectivity analysis
            sel_idx = (condition_df['retain']==True) | (condition_df['update']==True)
            sel_spike_rate = spike_rate[sel_idx][:,area_df['PFC']]
            sel_label = label_data[:,sel_idx]
            n_neurons = sel_spike_rate.shape[1]

            # chosen and unchosen labels
            chosen_idx = (sel_label[2]<sel_label[3]).astype(int)
            unchosen_idx = (sel_label[2]>sel_label[3]).astype(int)
            trial_idx = np.arange(sel_label.shape[1])
            chosen_label = sel_label[chosen_idx,trial_idx]
            unchosen_label = sel_label[unchosen_idx,trial_idx]

            start_t = t_to_idx(2.6)
            end_t = t_to_idx(3.2)
            n_time = int(end_t-start_t)

            z_spike_rate = z_score(sel_spike_rate)
            coef_array = np.zeros((n_neurons, 6))
            pvalue_array = np.zeros((n_neurons, 6))

            for n in range(n_neurons):
                df = pd.DataFrame({
                    'r': np.mean(z_spike_rate[:,n,start_t:end_t], axis=-1),
                    'A': chosen_label,
                    'B': unchosen_label
                })

                ols_model = smf.ols('r ~ C(A) + C(B)', data=df).fit()
                pvalue_array[n] = ols_model.pvalues.iloc[1:]
                coef_array[n] = ols_model.params.iloc[1:]

            sig_chosen_neurons = (pvalue_array[:,:3]<0.05).any(axis=-1)
            sig_unchosen_neurons = (pvalue_array[:,3:]<0.05).any(axis=-1)

            mix_idx = np.where(sig_chosen_neurons & sig_unchosen_neurons)[0]
            chosen_idx = np.where(sig_chosen_neurons)[0]
            unchosen_idx = np.where(sig_unchosen_neurons)[0]

            # save data
            np.savez(f'/Users/huidili/Desktop/wmupdate/analysis_result/selectivity/{subject}/post_decision/{session_name}_ols_params_pvalues.npz', params=coef_array, pvalues=pvalue_array)
            np.savez(f'/Users/huidili/Desktop/wmupdate/analysis_result/selectivity/{subject}/post_decision/{session_name}_selective_neuron_index', n_neurons=n_neurons, chosen=chosen_idx, unchosen=unchosen_idx, mix=mix_idx)

080311Tir_area_df.pkl
080811Tir_area_df.pkl
061311Tir_area_df.pkl
062911Tir_area_df.pkl
072511Tir_area_df.pkl
062211Tir_area_df.pkl
080211Tir_area_df.pkl
070111Tir_area_df.pkl
071511Tir_area_df.pkl
080911Tir_area_df.pkl
072811Tir_area_df.pkl
080111Tir_area_df.pkl
080511Tir_area_df.pkl
070611Tir_area_df.pkl
072711Tir_area_df.pkl
081111Tir_area_df.pkl
070811Tir_area_df.pkl
072911Tir_area_df.pkl
072211Tir_area_df.pkl
071811Tir_area_df.pkl
080411Tir_area_df.pkl
081011Tir_area_df.pkl
061411Tir_area_df.pkl
